In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

# 1. Convert Numpy arrays to PyTorch Tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)  # shape (N,1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)



In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)



In [ ]:
# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [ ]:
# 4. Print shape of one batch
for images, labels in train_loader:
    print("Batch images shape:", images.shape)
    print("Batch labels shape:", labels.shape)
    break


In [ ]:
# 5. Display sample images
def show_images(dataset, num_images=5):
    fig, axes = plt.subplots(1, num_images, figsize=(15, 3))
    for i in range(num_images):
        img = dataset[i][0].permute(1, 2, 0).numpy()  # convert back to HWC
        age = dataset[i][1].item()
        axes[i].imshow(img)
        axes[i].set_title(f"Age: {age}")
        axes[i].axis("off")
    plt.show()

show_images(train_dataset)

In [ ]:
import torch.nn as nn
import torch.optim as optim

# Task 1: Model class with 4 linear layers
class AgePredictor(nn.Module):
    def __init__(self):
        super(AgePredictor, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(3*X_train.shape[2]*X_train.shape[3], 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)  # regression output
        return x

In [ ]:
# Task 2: Training loop
def train_loop(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    return running_loss / len(loader)

In [ ]:
# Task 3: Validation loop
def val_loop(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
    return running_loss / len(loader)

In [ ]:
# Task 4: Define device, model, loss, optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AgePredictor().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Task 5: Train for 20 epochs
train_losses, val_losses = [], []
epochs = 20

for epoch in range(epochs):
    train_loss = train_loop(model, train_loader, criterion, optimizer, device)
    val_loss = val_loop(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

In [ ]:
# Task 1: Plot training and validation loss
plt.figure(figsize=(8,5))
plt.plot(range(1, epochs+1), train_losses, label="Train Loss")
plt.plot(range(1, epochs+1), val_losses, label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss (MSE)")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

In [ ]:
# Task 2 (Bonus): Show predictions vs actual ages
model.eval()
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)
with torch.no_grad():
    preds = model(images).cpu().numpy()

fig, axes = plt.subplots(1, 5, figsize=(15,3))
for i in range(5):
    img = images[i].cpu().permute(1,2,0).numpy()
    actual = labels[i].item()
    predicted = preds[i][0]
    axes[i].imshow(img)
    axes[i].set_title(f"Actual: {actual}\nPred: {predicted:.1f}")
    axes[i].axis("off")
plt.show()